In [ ]:
#| hide
#| eval: false
! [ -e /content ] && pip install -Uqq xcube  # upgrade xcube on colab

In [ ]:
#| export
# from fastai.torch_imports import *
# from fastai.torch_core import *
# from fastai.callback.core import *
# from fastcore.all import *
from xcube.imports import *
from datasets import load_dataset, load_from_disk 
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp text.topicmodel

# Topic Modeling
> with the excellent library BERTopic

In this notebook we perform topic modeling with the python library [BERTopic](https://maartengr.github.io/BERTopic/index.html)

In [ ]:
#| hide
from huggingface_hub import notebook_login, logout
notebook_login()

In [ ]:
#| export
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech

from bertopic.vectorizers import ClassTfidfTransformer
import openai

In [ ]:
#| export
def get_topic_model(embedding_model="all-MiniLM-L6-v2",
                    umap_n_neighbors=15, umap_n_components=5, umap_min_dist=0.0, umap_metric='cosine',
                    hdbscan_min_cluster_size=15, hdbscan_metric='euclidean', hdbscan_cluster_selection_method='eom',
                    vectorizer_stop_words="english",
                    representation_model=None,
                    random_state=None,
                    low_memory=False,
                    openai_API_KEY=None,
                    **kwargs):
    
    # Step 1 - Extract embeddings
    embedding_model = SentenceTransformer(embedding_model)
    
    # Step 2 - Reduce dimensionality
    umap_model = UMAP(n_neighbors=umap_n_neighbors, n_components=umap_n_components, min_dist=umap_min_dist, metric=umap_metric, random_state=random_state, low_memory=low_memory)

    # Step 3 - Cluster reduced embeddings
    hdbscan_model = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size, metric=hdbscan_metric, cluster_selection_method=hdbscan_cluster_selection_method, prediction_data=True)

    # Step 4 - Tokenize topics
    vectorizer_model = CountVectorizer(stop_words=vectorizer_stop_words, min_df=2, ngram_range=(1,2))

    # Step 5 - Create topic representation
    ctfidf_model = ClassTfidfTransformer()

    # Step 6 - (Optional) Fine-tune topic representations with a `bertopic.representation` model
    # KeyBERT
    keybert_model = KeyBERTInspired()

    # GPT-3.5
    if openai_API_KEY:
        client = openai.OpenAI(api_key=openai_API_KEY)
        prompt = """
        I have a topic that contains the following documents: 
        [DOCUMENTS]
        The topic is described by the following keywords: [KEYWORDS]

        Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
        topic: <topic label>
        """
        openai_model = OpenAI(client, model="gpt-4o-mini", exponential_backoff=True, chat=True, prompt=prompt)
    representation_model = {
        "KeyBERT": keybert_model,
        # "OpenAI": openai_model if openai_API_KEY else None,  # Uncomment if you will use OpenAI
    }

    # Conditionally add OpenAI model if the API key is available
    if openai_API_KEY:
        representation_model["OpenAI"] = openai_model
    
    # All steps together
    topic_model = BERTopic(embedding_model=embedding_model,
                           umap_model=umap_model,
                           hdbscan_model=hdbscan_model,
                           vectorizer_model=vectorizer_model,
                           ctfidf_model=ctfidf_model,
                           representation_model=representation_model,
                           verbose=True,
                           low_memory=low_memory,
                           **kwargs
                           )

    return embedding_model, topic_model
 

## Data

Get the data to perform topic modeling on:

In [ ]:
trn_dset = load_dataset("deb101/lf_amazon_131k", split='train')
# trn_dset = load_from_disk(f"{Path.cwd().parent}/scripts/temp/lf_amazon_131k") # to load locally
trn_dset

Dataset({
    features: ['uid', 'title', 'content', 'target_ind', 'target_rel'],
    num_rows: 294805
})

In [ ]:
splt_sz = 2000
titles = trn_dset['title'][:splt_sz]
contents = trn_dset['content'][:splt_sz]
docs = [f'{t} \n {c}' for t,c in  zip(titles, contents)]
# docs = [sent_tokenize(doc) for doc in docs]
# docs = [sentence for doc in docs for sentence in doc]
len(docs)

2000

## Training

In [ ]:
embedding_model, topic_model = get_topic_model(embedding_model="all-MiniLM-L6-v2", 
                                                       openai_API_KEY=None, 
                                                       language='multilingual', 
                                                       calculate_probabilities=False,
                                                       low_memory=False,
                                                       seed_topic_list=None)

In [ ]:
%%time
# Pre-calculate embeddings
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

CPU times: user 3.78 s, sys: 458 ms, total: 4.24 s
Wall time: 2.21 s


In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings) # topics and the corresponding probabilities

2024-09-23 10:59:19,327 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-09-23 10:59:23,427 - BERTopic - Dimensionality - Completed ✓
2024-09-23 10:59:23,428 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-09-23 10:59:23,463 - BERTopic - Cluster - Completed ✓
2024-09-23 10:59:23,466 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-09-23 10:59:24,444 - BERTopic - Representation - Completed ✓


In [ ]:
topic_model.topic_labels_

{-1: '-1_music_christmas_songs_new',
 0: '0_book_new_text_edition',
 1: '1_inch_high_use_quality',
 2: '2_album_music_band_rock',
 3: '3_music_piano_opera_composer',
 4: '4_yoga_workout_body_fitness'}

In [ ]:
probs.shape

(2000,)

In [ ]:
# Create topic model
topic_model = BERTopic(calculate_probabilities=True).fit(docs)
topics, probs = topic_model.transform(docs)

In [ ]:
topic_model.get_topics().keys()

dict_keys([-1, 0, 1, 2, 3, 4, 5, 6, 7, 8])

In [ ]:
len(topics), topic_model.probabilities_[999], topics[999]

(1000,
 array([0.02175438, 0.01410349, 0.04044805, 0.14069893, 0.06586901,
        0.05766029, 0.07983988, 0.08660192, 0.05492531]),
 -1)

In [ ]:
topic_model.probabilities_[789], topics[789]

(array([0.07337232, 0.92662768]), 1)

In [ ]:
outlier_count = df_topics[df_topics['Topic']==-1].Count.values[0]
outlier_count

138075

In [ ]:
df_topics = topic_model.get_topic_info()
df_topics.head()

,Topic,Count,Name,Representation,KeyBERT,Representative_Docs
0,-1,138075,-1_love_life_story_album,"[love, life, story, album, like, book, edition, refers, text refers, text]","[novel, book, author, books, readers, characters, stories, writer, read, edition title]","[In Like Flynn: A Molly Murphy Mystery \n In Bowen's absorbing, well-plotted fourth entry in this Agatha Awardwinning historical series (after 2003's For the Love of Mike), Molly Murphy's former beau, policeman Daniel Sullivan, arranges for Molly to leave New York City (and the rapidly spreading typhoid epidemic of 1902). The police are interested in the Sorensons, a pair of sisters working as spiritualists, whom they want to expose as fakes. Molly joins the upstate household of Sen. Barney Flynn, posing as one of his numerous cousins recently arrived from Ireland. Flynn's wife, Theresa, h..."
1,0,1729,0_metal_band_punk_bands,"[metal, band, punk, bands, album, death metal, rock, hardcore, heavy metal, vocals]","[black sabbath, death metal, metal band, metallica, metal bands, thrash metal, black metal, new album, debut album, punk rock]","[The Unreal Never Lived \n Hailing from the ever-gray skies of Eugene, Oregon comes YOB, a band already well on their way to make their mark in the DOOM metal scene worldwide. Formed in 1996 by founding member Mike Scheidt YOB started creating and destroying ideas that would become the foundation for one of DOOM metals most shining hopefuls. YOB is prepared to release their sophomore effort on Metal Blade Records titled The Unreal Never Lived, which follows up their tremendously successful Metal Blade debut The Illusion Of Motion. If the bands previous effort didnt catch your attention the..."
2,1,1618,1_jazz_coltrane_ellington_saxophonist,"[jazz, coltrane, ellington, saxophonist, pianist, tenor, trumpeter, bop, drummer, bassist]","[john coltrane, modern jazz, coltrane, jazz history, jazz, duke ellington, saxophonist, jazz icons, ellington, tenor saxophonist]","[Duke Ellington & John Coltrane \n Perhaps looking to renew his inspiration or maybe simply wanting to broaden his horizons, Duke Ellington began a string of collaborations in the second half of his career--whereas before that, his own band was stimulus enough. Whatever the reason, almost all of his collaborations succeeded at high levels, although none of his shared sessions are more intriguing on the surface than this 1962 date with the preeminent sax star of the day. In reality, the record amounts to ""Coltrane Plays Ellington"" (plus one Coltrane original) because the tenor man is the wh..."
3,2,1431,2_cd album_cd_audio cd_album,"[cd album, cd, audio cd, album, audio, album cd, record album, cd cd, music audio, cd single]","[cd album, album cd, music cd, cd, cd featuring, uk cd, cd cd, cd like, record album, cd single]","[Fairlytales \n CD ALBUM, Something/Anything? \n CD ALBUM, Three \n CD ALBUM]"
4,3,1272,3_cassette_audio cassette_cassette tape_tape,"[cassette, audio cassette, cassette tape, tape, audio, cassette cassette, sealed audio, cassette best, cassette live, cassettes]","[cassette, audio cassette, cassettes, cassette cassette, music cassette, cassette audio, cassette collection, audio cassettes, cassette player, cassette world]","[Rechordings \n CASSETTE., Moodfood \n CASSETTE, Stereotomy \n CASSETTE]"


In [ ]:
#| export
def get_topicsmap(df_topics):
    """
    Creates a dictionary mapping topics to their corresponding names.

    Args:
    - df_topics (pandas.DataFrame): A DataFrame containing topic information.
        It should have at least two columns: 'Topic' and 'Name'.

    Returns:
    - dict: A dictionary where the keys are topic numbers and the values are
        the corresponding names of the topics.
    """
    return dict(zip(df_topics['Topic'], df_topics['CustomName']))


In [ ]:
# Example usage
topics_map = get_topicsmap(df_topics)
topics_map

In [ ]:
#| export
def print_representative_topics(df, topic_num):
    """
    Print representative topics for a specified topic number.

    Args:
        df (DataFrame): The pandas DataFrame containing the topics and representative documents, obtained by `topic_model.get_topic_info()`.
        topic_num (int): The topic number for which representative topics should be printed.

    Returns:
        None

    Example:
        >>> print_representative_topics(df, 2)
    """
    filtered_topic = df[df['Topic'] == topic_num].iloc[0]
    print(f"Representative docs for Topic {topic_num}, that is \"{filtered_topic['Name']}\":")
    for doc in filtered_topic['Representative_Docs']:
        print(doc)
        print("****************")


In [ ]:
# Example usage:
topic_num = 3
print_representative_topics(df_topics, topic_num)


Representative docs for Topic 3, that is "3_14 hrs_13_14_rated":
From the Producers of the Academy-Award nominated film, WAR/DANCE and Executive Producer Eva Longoria, this award-winning documentary provides an intimate glimpse into the lives of these children who struggle to dream while working 12 14 hours a day, 7 days a week to feed America.
****************
Storm Of The Century - Rated PG-13 - 256 Mins.
****************
Stephen King's - Rose Red, Desperation, Storm Of The Century, Riding The Bullet 
 Rose Red - Rated PG-13 - 254 Mins.
****************


In [ ]:
#| export
def get_topic(topic_model, topic_num):
    print('\n'.join(L(topic_model.get_topic(topic_num, full=False)).map(lambda pair: f"{pair[0]} --> {pair[1]}")))

In [ ]:
# Example usage
get_topic(topic_model, topic_num)

14 hrs --> 0.5073680877685547
13 --> 0.48660773038864136
14 --> 0.4783494174480438
rated --> 0.4772558808326721
minutes --> 0.4752909541130066
12 --> 0.46001094579696655
15 --> 0.45952650904655457
14 pc --> 0.4246455430984497
24 --> 0.4224071204662323
easy minutes --> 0.41836634278297424


Look at a random document and figure if it belongs to the correct topic:

In [ ]:
df_docs = topic_model.get_document_info(tokenized_docs)
df_docs.sample(1)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
4427,"Reduced to working at a gas station and drowning his sorrows at the local pub, Harry is shocked to get a phone call informing him that he has a 34-year-old son: one David Venning, a brilliant mathematician who lies comatose in hospital.",94,94_harry barnett_barnett_harry_robert goddard,"[harry barnett, barnett, harry, robert goddard, goddard, entangled, disappearance, coma, son knew, blue]","[In this above-average thriller by Goddard (A Debt of Dishonour, LJ 12/91), the presumably childless Harry Barnett, living a quiet, aimless life in Britain, receives an anonymous call informing him that his son, a brilliant mathematician, is comatose., Harry Barnett is shocked to learn that he has a son--David Venning, a brilliant mathematician, now languishing in hospital in a diabetic coma.And this is only the first and smallest of the mysteries he is about to encounter.David's condition is attributed to an accident or suicide attempt.But Harry discovers that his mathematical notebooks a...",harry barnett - barnett - harry - robert goddard - goddard - entangled - disappearance - coma - son knew - blue,0.916249,False


## Search Topics:

Create some dummy product descriptions to search for most relevant topics:

In [ ]:
product_description = """
Introducing our latest collection of men's casual shirts, designed to elevate your everyday style with effortless sophistication. Crafted from premium quality cotton fabric, our shirt offers unrivaled comfort and breathability, making it perfect for all-day wear.

Featuring a classic button-down design with a modern twist, this shirt combines timeless elegance with contemporary flair. The slim-fit silhouette ensures a sleek and tailored look, while the versatile color options make it easy to pair with any outfit.

Whether you're heading to the office, meeting friends for brunch, or enjoying a night out on the town, this shirt will keep you looking and feeling your best. Dress it up with tailored trousers and loafers for a polished ensemble, or keep it casual with jeans and sneakers for a laid-back vibe.

Upgrade your wardrobe with our stylish men's shirt and experience the perfect blend of comfort, quality, and style. Add it to your cart today and elevate your wardrobe essentials with ease.
"""

album_description = """
Introducing our latest music album, an eclectic blend of captivating melodies and soul-stirring rhythms that will take you on a musical journey like no other. With a unique fusion of genres ranging from jazz and blues to funk and soul, this album promises to delight music lovers of all tastes.

Featuring mesmerizing vocals, skillful instrumentation, and heartfelt lyrics, each track on this album tells a story and evokes a range of emotions. From upbeat anthems that will have you dancing along to heartfelt ballads that will tug at your heartstrings, there's something for everyone on this album.

Whether you're looking to unwind after a long day, set the mood for a romantic evening, or simply groove to some infectious beats, this album has you covered. Let the music transport you to new and exciting places, and allow yourself to get lost in its enchanting melodies.

Experience the magic of our latest music album and discover why it's destined to become a timeless classic. Add it to your playlist today and prepare to be captivated from the very first note.
"""

glove_description = """
GripMaster Elite Baseball Gloves Experience unmatched comfort and control with the GripMaster Elite Baseball Gloves. 

Designed for the serious player, these gloves are made from high-quality leather, ensuring durability and a soft, supple feel. The tailored fit adapts to your hand shape, enhancing flexibility and grip. 

Strategic perforations increase breathability, keeping your hands cool under pressure. 

Whether you're catching fastballs or fielding grounders, the GripMaster Elite provides the support and tactile sensitivity you need to perform at your best.
"""

In [ ]:
#| export
def print_top_topics(topic_model, description, topics_map, top_n=5):
    """
    Prints the top N topics related to a given description.

    Args:
    - topic_model (BERTopic): A BERTopic model instance.
    - description (str): The description for which top topics are to be found.
    - topics_map (dict): A dictionary mapping topic numbers to their corresponding names.
    - top_n (int, optional): The number of top topics to print. Defaults to 5.

    Returns:
    None
    """
    similar_topics, similarity = topic_model.find_topics(description, top_n=top_n)
    print(f'The top {top_n} topics for the description are:')
    for i, (topic, sim) in enumerate(zip(similar_topics, similarity)):
        print(f"{i}: {topics_map.get(topic, None)}, similarity is {sim}")
        get_topic(topic_model, topic)
        print("**********************")


In [ ]:
# Example usage
print_top_topics(topic_model, glove_description, topics_map)

The top 5 topics for the description are:
0: 95_world series_inning_dodgers_game 2008, similarity is 0.2928817570209503
world series --> 0.6577011346817017
inning --> 0.5309538841247559
dodgers --> 0.46776020526885986
game 2008 --> 0.44428202509880066
philadelphia --> 0.4256771504878998
2008 --> 0.37799695134162903
memorabilia collection --> 0.3251769542694092
teams --> 0.31967663764953613
collector edition --> 0.31944572925567627
howard --> 0.3033175468444824
**********************
1: 2_bits_accessories_socket_tools, similarity is 0.28658172488212585
bits --> 0.43933504819869995
accessories --> 0.3825658857822418
socket --> 0.380443811416626
tools --> 0.36953914165496826
hardware --> 0.35977843403816223
12 inch --> 0.35811662673950195
blade --> 0.3501020073890686
drill --> 0.34510794281959534
tool --> 0.344309002161026
inch --> 0.3441663384437561
**********************
2: 6_shirt clothing_cotton shirt_clothing_shirt, similarity is 0.25819966197013855
shirt clothing --> 0.5779263377189

In [ ]:
topic_model.topic_aspects_

{}

## Serialization

In [ ]:
#| export
def save_topic_model(topic_model, path, embedding_model_name=None):
    """
    Saves the topic model to the specified path, including the embedding model.

    Args:
    - topic_model (BERTopic): A BERTopic model instance.
    - path (str): The path where the model will be saved.
    - embedding_model: The embedding model used in the topic model.

    Returns:
    None
    """
    if embedding_model_name: print(f"Saving the embedding model {embedding_model_name} along with topic_model.")
    topic_model.save(path, serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model_name)

In [ ]:
#| eval: false 
import tempfile
with tempfile.TemporaryDirectory() as tmp_dir:
        # Define a dummy topic model and embedding model
        
        # Call the save_topic_model function with the temporary directory path
        save_topic_model(topic_model, tmp_dir, embedding_model)
        print('Saved')

        files = os.listdir(tmp_dir) 
        print(files)
        
        # Check if the model files are saved in the temporary directory
        for f in files:
                assert(os.path.exists(os.path.join(tmp_dir, f)))

Saved
['ctfidf.safetensors', 'config.json', 'topic_embeddings.safetensors', 'ctfidf_config.json', 'topics.json']


In [ ]:
#| eval: false
#| hide
# Make a temp directory and save it so save it so that we can load later
model_dir = Path.cwd()/'tmp/lf_amazon_131k'
model_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
save_topic_model(topic_model, model_dir, embedding_model)

In [ ]:
# Load the saved topic_model with the embedding model that was used
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
loaded_model = BERTopic.load(model_dir, embedding_model=embedding_model)
topics_map = get_topicsmap(loaded_model.get_topic_info())
topic = 0
print(f"Loading saved topicmodel to check... Printing topic {topic}, that is \"{topics_map[topic]}\"")
get_topic(loaded_model, topic)

Loading saved topicmodel to check... Printing topic 0, that is "0_viton_heat_kit_steel"
viton --> 0.46385058760643005
heat --> 0.20553384721279144
kit --> 0.1972728669643402
steel --> 0.19067609310150146
oil --> 0.18439994752407074
designed --> 0.17696812748908997
plastic --> 0.17308196425437927
durometer --> 0.17277032136917114
leather --> 0.16903001070022583
color --> 0.16852779686450958


## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()